In [4]:
import pandas as pd
from matminer.featurizers.structure import SiteStatsFingerprint, BagofBonds
from pymatgen.core import Structure 

# 1. Load dataset containing pymatgen Structure objects
# Note: If this file is in your Downloads folder, change the path to 'C:/Users/HP/Downloads/perovskite_structures.pkl'
# Change this back to the .pkl file!
file_path = r"C:\Users\HP\Downloads\perovskite_structures.pkl" 
df = pd.read_pickle(file_path)
print(f"Loaded {len(df)} crystal structures.")

# Safely identify if there are tracking columns (like formula or ID) to keep in the final CSV
id_cols = [col for col in ['material_id', 'formula', 'target'] if col in df.columns]

# 2. SiteStatsFingerprint
# Computes statistics (mean, std, etc.) of local site properties
print("Computing SiteStatsFingerprint features (this may take a moment)...")
ssf = SiteStatsFingerprint.from_preset("CoordinationNumber_ward-prb-2017")
# Added ignore_errors=True so a single bad structure doesn't crash the whole script
df_ssf = ssf.featurize_dataframe(df[['structure']], col_id='structure', ignore_errors=True)
print(f"Generated {df_ssf.shape[1] - 1} SiteStatsFingerprint features.")

# 3. Bag of Bonds (BoB)
# Represents a structure by the distribution of its bond lengths
print("Computing BagofBonds features...")
bob = BagofBonds()
bob.fit(df['structure']) # BoB requires fitting to the dataset to determine bonding pairs
df_bob = bob.featurize_dataframe(df[['structure']], col_id='structure', ignore_errors=True)
print(f"Generated {df_bob.shape[1] - 1} BagofBonds features.")

# 4. Merge and Save the Structural Feature Matrix
# Concatenate the generated features, dropping the heavy 'structure' objects
df_features = pd.concat([
    df_ssf.drop(columns=['structure']), 
    df_bob.drop(columns=['structure'])
], axis=1)

# Re-attach the identifiers (like formula) so you know which row belongs to which material
if id_cols:
    df_structural = pd.concat([df[id_cols], df_features], axis=1)
else:
    df_structural = df_features

# Save the final output
out_file = 'structural_features_s22.csv'
df_structural.to_csv(out_file, index=False)
print(f"✅ Structural feature matrix successfully generated and saved to '{out_file}'.")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\HP\\Downloads\\perovskite_structures.pkl'